# 01. Load, validate and clean

First step for every new export. It reads the CSVs, runs every validation check, and shows exactly what the cleaning step removes. **Dummy data by default.**

Checks are *hard* (the analysis refuses to run: the derived numbers would be untrustworthy) or *soft* (reported and handled explicitly). Try `DATA_DIR = ROOT / 'data' / 'dummy_dirty'` below to see the validator catch 8 injected faults.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA_DIR = ROOT / "data" / "dummy"      # <- point this at data/raw/<your export folder> for real data
OUT_DIR = ROOT / "outputs"

from pipeline.io import load_dataset
from pipeline.validate import validate
from pipeline.clean import clean

ds = load_dataset(DATA_DIR)
print({"events": len(ds.events), "trials": ds.events.trial_id.nunique(),
       "participants": ds.participants.participant_id.nunique(), "excluded in exclusions.csv": len(ds.exclusions)})
ds.events.head(10)

{'events': 817, 'trials': 105, 'participants': 15, 'excluded in exclusions.csv': 1}


,participant_id,trial_id,sequence,block_position,trial_order,pattern,task_id,task_pair,target_node,min_selections,t_ms,event_type,node_id,current_node_after,on_path
0,P01,P01_PR1,S1,0,0,DH,PR1,PR,PR1.2.1,3,0,start,,ROOT,1
1,P01,P01_PR1,S1,0,0,DH,PR1,PR,PR1.2.1,3,510,menu_open,,ROOT,1
2,P01,P01_PR1,S1,0,0,DH,PR1,PR,PR1.2.1,3,2313,select,PR1,PR1,1
3,P01,P01_PR1,S1,0,0,DH,PR1,PR,PR1.2.1,3,4649,select,PR1.2,PR1.2,1
4,P01,P01_PR1,S1,0,0,DH,PR1,PR,PR1.2.1,3,6795,select,PR1.2.1,PR1.2.1,1
5,P01,P01_PR1,S1,0,0,DH,PR1,PR,PR1.2.1,3,6795,target_reached,PR1.2.1,PR1.2.1,1
6,P01,P01_T1,S1,1,1,DH,T1,A,C1.1.2,3,0,start,,ROOT,1
7,P01,P01_T1,S1,1,1,DH,T1,A,C1.1.2,3,1050,menu_open,,ROOT,1
8,P01,P01_T1,S1,1,1,DH,T1,A,C1.1.2,3,3116,select,C1,C1,1
9,P01,P01_T1,S1,1,1,DH,T1,A,C1.1.2,3,5005,select,C1.1,C1.1,1


## Validation report

In [2]:
report = validate(ds)
r = report.to_frame()
r

,check,severity,passed,n_issues,message
0,schema,hard,True,0,required columns and values present
1,one_start,hard,True,0,every trial has one start at t = 0
2,one_terminal_event,hard,True,0,every trial ends with exactly one target_reach...
3,monotonic_time,hard,True,0,t_ms non-decreasing in every trial
4,time_limit,hard,True,0,all events within 0-180000 ms
5,nodes_in_tree,hard,True,0,every node id exists in tree.csv
6,targets_match_tasks,hard,True,0,trial targets and pairs match tasks.csv
7,min_selections_match_pattern,hard,True,0,min_selections is 2 for MM and 3 for DH/ND
8,trials_per_participant,soft,True,0,every participant has 6 experimental trials
9,participants_per_sequence,soft,True,0,"15 participants, 5 per sequence"


In [3]:
print('HARD failures:', [x.name for x in report.hard_failures])
print('soft failures:', [x.name for x in report.soft_failures])

HARD failures: []
soft failures: []


### Offending rows
Empty when everything passes. Fix hard failures in the raw export (and note the fix in the cleaning log) before continuing.

In [4]:
report.details_frame()

""


## Cleaning
Only three things are ever removed, each logged: exact duplicate event rows, practice trials, and trials listed in `exclusions.csv` (device/app faults, decided without looking at the pattern). **No statistical outlier removal.**

In [5]:
report.raise_if_hard()        # stops here if the data has hard problems
clean_ds, log = clean(ds)
print(f'{ds.events.trial_id.nunique()} trials before, {clean_ds.events.trial_id.nunique()} after')
log

105 trials before, 89 after


,step,trial_id,n_event_rows,reason
0,practice,P01_PR1,6,"practice trial, not analysed"
1,practice,P02_PR1,5,"practice trial, not analysed"
2,practice,P03_PR1,12,"practice trial, not analysed"
3,practice,P04_PR1,6,"practice trial, not analysed"
4,practice,P05_PR1,8,"practice trial, not analysed"
5,practice,P06_PR1,10,"practice trial, not analysed"
6,practice,P07_PR1,8,"practice trial, not analysed"
7,practice,P08_PR1,7,"practice trial, not analysed"
8,practice,P09_PR1,6,"practice trial, not analysed"
9,practice,P10_PR1,8,"practice trial, not analysed"


In [6]:
log.groupby('step').agg(trials=('trial_id', 'nunique'), event_rows=('n_event_rows', 'sum'))

,trials,event_rows
step,,
exclusion,1,5
practice,15,104
